## Text Classification - Transformer (self-attention + norm + projection + skip-connection)

In [1]:
import torch
import torch.nn as nn
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace
 
corpus = [
    "gậy ông đập lưng ông",
    "có làm mới có ăn"
]
data_size = len(corpus)

# 0: negative - 1: positive
labels = [0, 1]

# Define the max vocabulary size and sequence length
vocab_size = 8
sequence_length = 5

In [2]:
# Initialize the tokenizer and define a trainer
tokenizer = Tokenizer(WordLevel())
tokenizer.pre_tokenizer = Whitespace()
tokenizer.enable_padding(pad_id=1, 
                         pad_token="<pad>", 
                         length=sequence_length)
tokenizer.enable_truncation(max_length=sequence_length)

# Train the tokenizer on your corpus
trainer = WordLevelTrainer(vocab_size=vocab_size, 
                           special_tokens=["<unk>", "<pad>"])
tokenizer.train_from_iterator(corpus, trainer)

# Tokenize and numericalize your samples
def vectorize(sentence, tokenizer):
    output = tokenizer.encode(sentence)
    return torch.tensor(output.ids, dtype=torch.long)

# Vectorize the samples
corpus_ids = []
for sentence in corpus:
    corpus_ids.append(vectorize(sentence, tokenizer))

In [3]:
for v in corpus_ids:
    print(v)

tensor([4, 3, 0, 6, 3])
tensor([2, 5, 7, 2, 0])


In [4]:
class TextCls_Model(nn.Module):
    def __init__(self, vocab_size, num_classes, embed_dim, num_heads, proj_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim=embed_dim, 
                                          num_heads=num_heads,
                                          batch_first=True) 
        self.norm1 = nn.LayerNorm(normalized_shape=embed_dim)      

        self.projection = nn.Linear(in_features=embed_dim, out_features=proj_dim)
        self.norm2 = nn.LayerNorm(normalized_shape=proj_dim)      

        self.flatten = nn.Flatten()       
        self.classifier = nn.Linear(in_features=proj_dim*5, out_features=2) 

    def forward(self, x):
        x = self.embedding(x)
        
        # a transformer block
        x_saved = x
        x,_ = self.attn(x, x, x)
        x = self.norm1(x_saved+x)    # skip-connection

        x_saved = x
        x = self.projection(x)
        x = self.norm2(x_saved+x)    # skip-connection

        x = self.flatten(x)
        x = self.classifier(x)
        return x

num_classes = 2
model = TextCls_Model(vocab_size, num_classes, 10, 1, 10)

In [5]:
input_1 = torch.tensor([[4, 3, 0, 6, 3]], dtype=torch.long)
label_1 = torch.tensor([0], dtype=torch.long)

output = model(input_1)
print(output.shape)

torch.Size([1, 2])


In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(),
                             lr=0.1)

## Train

In [7]:
inputs = torch.tensor([[4, 3, 0, 6, 3],
                       [2, 5, 7, 2, 0]], dtype=torch.long)
labels = torch.tensor([0, 1], dtype=torch.long)

for _ in range(50):
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    print(loss)

    loss.backward()
    optimizer.step()

tensor(0.5163, grad_fn=<NllLossBackward0>)
tensor(0.0509, grad_fn=<NllLossBackward0>)
tensor(0.0371, grad_fn=<NllLossBackward0>)
tensor(0.0294, grad_fn=<NllLossBackward0>)
tensor(0.0243, grad_fn=<NllLossBackward0>)
tensor(0.0208, grad_fn=<NllLossBackward0>)
tensor(0.0181, grad_fn=<NllLossBackward0>)
tensor(0.0161, grad_fn=<NllLossBackward0>)
tensor(0.0145, grad_fn=<NllLossBackward0>)
tensor(0.0131, grad_fn=<NllLossBackward0>)
tensor(0.0120, grad_fn=<NllLossBackward0>)
tensor(0.0111, grad_fn=<NllLossBackward0>)
tensor(0.0103, grad_fn=<NllLossBackward0>)
tensor(0.0096, grad_fn=<NllLossBackward0>)
tensor(0.0090, grad_fn=<NllLossBackward0>)
tensor(0.0085, grad_fn=<NllLossBackward0>)
tensor(0.0080, grad_fn=<NllLossBackward0>)
tensor(0.0075, grad_fn=<NllLossBackward0>)
tensor(0.0072, grad_fn=<NllLossBackward0>)
tensor(0.0068, grad_fn=<NllLossBackward0>)
tensor(0.0065, grad_fn=<NllLossBackward0>)
tensor(0.0062, grad_fn=<NllLossBackward0>)
tensor(0.0059, grad_fn=<NllLossBackward0>)
tensor(0.00

## Check if the model has adready learned sucessfuly

In [8]:
outputs = model(inputs)
print(outputs)
print(torch.softmax(outputs, axis=-1))

tensor([[ 2.6159, -3.2829],
        [-3.5020,  2.4382]], grad_fn=<AddmmBackward0>)
tensor([[0.9973, 0.0027],
        [0.0026, 0.9974]], grad_fn=<SoftmaxBackward0>)
